In [1]:
import pandas as pd
import holidays
from pathlib import Path
import sys
sys.path.append("../machine_learning")
from forecast_checks import LeakageChecker

# Goal #
add the following features
* weekday [mon, tues, ...]
* weekend [boool]
* holiday [bool]
* lag features:
    * consumption_6days_ago
    * consumption_7days_ago
    * consumption_14days_ago
    * min_consumption_last_available_day
    * max_consumption_last_available_day

## Drop duplicate columns ##
* temperature_2m from vaerdata_2022_2024.csv
* wind_gust_10m (94% correlation with wind_speed_10m)

## Output ##
New CSV file containing all the data from feature engineeering, consumption_temp.csv and vaerdata_2022_2024.csv. This will be the *one* file used for ML

In [2]:
DATA_DIR = Path("../data")
CONSUMPTION_DATA = DATA_DIR / "consumption_temp.csv"
WEATHER_DATA = DATA_DIR / "vaerdata_2022_2024.csv"
OUTPUT = DATA_DIR / "dataset.csv"

LAG_DAYS = (6, 7, 14)
#The forecast for day D+1 is made on day D, and consumption is only known up to D-5: 6 days before the target day
LAST_AVAILABLE_DAY = 6
#temperature_2m duplicates the delivered temperature (0.99 correlation), wind_gusts_10m duplicates wind_speed_10m (0.94)
DUPLICATE_WEATHER_COLS = ["temperature_2m", "wind_gusts_10m"]

In [3]:
#Load both files with time as UTC so they can be joined on (time, location)
df = pd.read_csv(CONSUMPTION_DATA)
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df.sort_values(["time", "location"]).reset_index(drop=True)

weather = pd.read_csv(WEATHER_DATA)
weather["time"] = pd.to_datetime(weather["time"], utc=True)

print(df.info())
print(df["location"].value_counts())
print(df.iloc[0:3, :])
print(df.iloc[-1, :])

<class 'pandas.DataFrame'>
RangeIndex: 49494 entries, 0 to 49493
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   time         49494 non-null  datetime64[us, UTC]
 1   location     49494 non-null  str                
 2   consumption  49494 non-null  float64            
 3   temperature  49494 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(2), str(1)
memory usage: 1.5 MB
None
location
bergen         8641
oslo           8641
stavanger      8641
tromsø         8641
trondheim      8641
helsingfors    6289
Name: count, dtype: int64
                       time   location  consumption  temperature
0 2022-04-07 21:00:00+00:00     bergen     1.113325         -0.3
1 2022-04-07 21:00:00+00:00       oslo     4.092830          1.0
2 2022-04-07 21:00:00+00:00  stavanger     2.057858          1.3
time           2023-04-02 21:00:00+00:00
location                       trondheim
consu

Ingen null-verdier, men ser det er mindre data for Helsinki

In [4]:
#Helsingfors reports 0 MW for its first 32 hours: those hours were not delivered, so drop them
df = df[df["consumption"] > 0].reset_index(drop=True)
weather = weather.drop(columns=DUPLICATE_WEATHER_COLS)

In [5]:
class FeatureBuilder:
    """Adds calendar and consumption features that are known when the forecast is made (10:00 the day before)."""

    def __init__(self, df):
        self.df = df.copy()
        #Calendar effects follow the local clock, so date and hour are local (Finland is one hour ahead of Norway)
        oslo = self.df["time"].dt.tz_convert("Europe/Oslo").dt.tz_localize(None)
        helsinki = self.df["time"].dt.tz_convert("Europe/Helsinki").dt.tz_localize(None)
        self.is_finland = self.df["location"] == "helsingfors"
        local_time = oslo.where(~self.is_finland, helsinki)
        self.df["date"] = local_time.dt.normalize()
        self.df["hour"] = local_time.dt.hour

    def add_calendar_features(self):
        """Weekday (0 = Monday), weekend and national public holiday flags."""
        years = self.df["date"].dt.year.unique().tolist()
        norway = pd.to_datetime(list(holidays.Norway(years=years)))
        finland = pd.to_datetime(list(holidays.Finland(years=years)))
        self.df["weekday"] = self.df["date"].dt.weekday
        self.df["weekend"] = self.df["weekday"] >= 5
        self.df["holiday"] = (self.df["date"].isin(norway) & ~self.is_finland) | (self.df["date"].isin(finland) & self.is_finland)
        return self

    def add_lag_features(self, lag_days):
        """Consumption in the same local hour a given number of days before the target day."""
        #Mean per local hour because the October DST change repeats one hour
        hourly = self.df.groupby(["location", "date", "hour"], as_index=False)["consumption"].mean()
        for days in lag_days:
            lag = hourly.rename(columns={"consumption": f"consumption_{days}days_ago"})
            lag["date"] += pd.Timedelta(days=days)
            self.df = self.df.merge(lag, on=["location", "date", "hour"], how="left")
        return self

    def add_last_available_day_extremes(self, days):
        """Min and max consumption of the last day with metered data."""
        daily = self.df.groupby(["location", "date"], as_index=False)["consumption"].agg(
            min_consumption_last_available_day="min", max_consumption_last_available_day="max")
        daily["date"] += pd.Timedelta(days=days)
        self.df = self.df.merge(daily, on=["location", "date"], how="left")
        return self

In [6]:
#Build the features, then add the weather for the same hour
features = (FeatureBuilder(df)
            .add_calendar_features()
            .add_lag_features(LAG_DAYS)
            .add_last_available_day_extremes(LAST_AVAILABLE_DAY)
            .df)
dataset = features.merge(weather, on=["time", "location"], how="left")
dataset.head()

,time,location,consumption,temperature,date,hour,weekday,weekend,holiday,consumption_6days_ago,consumption_7days_ago,consumption_14days_ago,min_consumption_last_available_day,max_consumption_last_available_day,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m
0,2022-04-07 21:00:00+00:00,bergen,1.113325,-0.3,2022-04-07,23,3,False,False,NaN,NaN,NaN,NaN,NaN,1.2,0.0,0.84,3.47,12
1,2022-04-07 21:00:00+00:00,oslo,4.092830,1.0,2022-04-07,23,3,False,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.00,5.69,24
2,2022-04-07 21:00:00+00:00,stavanger,2.057858,1.3,2022-04-07,23,3,False,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.00,4.53,319
3,2022-04-07 21:00:00+00:00,tromsø,1.246582,-3.9,2022-04-07,23,3,False,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.00,1.71,97
4,2022-04-07 21:00:00+00:00,trondheim,1.970098,-2.8,2022-04-07,23,3,False,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.00,3.37,78


In [7]:
#Self-check: no rows lost or duplicated, every hour has weather, and features match the raw data
assert len(dataset) == len(df)
assert not dataset.duplicated(["location", "time"]).any()
assert dataset[weather.columns].notna().all().all()

oslo = dataset[dataset["location"] == "oslo"]
day = pd.Timestamp("2023-01-16")
week_before = day - pd.Timedelta(days=7)
last_available = day - pd.Timedelta(days=LAST_AVAILABLE_DAY)
target = oslo.query("date == @day and hour == 8").squeeze()
assert target["consumption_7days_ago"] == oslo.query("date == @week_before and hour == 8").squeeze()["consumption"]
assert target["min_consumption_last_available_day"] == oslo.query("date == @last_available")["consumption"].min()

holiday = dataset.groupby(["location", "date"])["holiday"].first()
assert holiday["oslo", pd.Timestamp("2022-05-17")]             #Norwegian Constitution Day
assert holiday["helsingfors", pd.Timestamp("2022-12-06")]      #Finnish Independence Day
assert not holiday["oslo", pd.Timestamp("2022-12-06")]
assert not holiday["oslo", pd.Timestamp("2022-09-18")]         #ordinary Sunday

#Missing lags: the first 6, 7 and 14 days of each series, plus the hour skipped at the March DST change
dataset.isna().sum()[lambda s: s > 0]

consumption_6days_ago                  870
consumption_7days_ago                 1014
consumption_14days_ago                2016
min_consumption_last_available_day     741
max_consumption_last_available_day     741
dtype: int64

In [ ]:
#Leakage check: scramble everything unknown when a sampled day is forecast (consumption after D-6, temperature after D)
#and verify that the features for that day do not change
LeakageChecker(df).check_features(lambda raw: (FeatureBuilder(raw)
                                               .add_calendar_features()
                                               .add_lag_features(LAG_DAYS)
                                               .add_last_available_day_extremes(LAST_AVAILABLE_DAY)
                                               .df))

In [8]:
dataset.to_csv(OUTPUT, index=False)
print(f"Wrote {len(dataset):,} rows and {dataset.shape[1]} columns to {OUTPUT}")

Wrote 49,462 rows and 19 columns to ../data/dataset.csv
